# 01 Explore: IBM AML transactions

Work card 01. Loads `data/HI-Small_Trans.csv`, checks memory, splits by time and looks at fraud rate and fan-in.

In [ ]:
import sys, time
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data import load_transactions, time_split, split_summary

!free -h
!nproc

Set `NROWS` to a number (e.g. `1_000_000`) if memory is tight. `None` loads everything.

In [ ]:
NROWS = None

start = time.time()
df = load_transactions(ROOT / 'data' / 'HI-Small_Trans.csv', nrows=NROWS)
print(f'Loaded {len(df):,} rows in {time.time() - start:.1f}s')
print(f'Memory: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB')
df.head()

In [ ]:
df.dtypes

## Time split

The test split must be the latest dates.

In [ ]:
train, val, test = time_split(df)
split_summary(train, val, test)

## Volume and fraud per day

Check whether the last days are very sparse. If so, the split may need adjusting.

In [ ]:
per_day = df.groupby(df['ts'].dt.date).agg(rows=('label', 'size'), fraud=('label', 'sum'))
per_day['fraud_rate_%'] = (100 * per_day['fraud'] / per_day['rows']).round(4)
per_day

## Currencies and payment formats

Amounts are in mixed currencies; card 02 must decide how to handle this.

In [ ]:
print(df['payment_currency'].value_counts().head(15))
print()
print(df.groupby('payment_format', observed=True)['label'].agg(['size', 'mean']).sort_values('mean', ascending=False))

## Fan-in on the training period

Receivers that collect from many different senders. Do they receive more laundering transactions?

In [ ]:
fan_in = train.groupby('dst').agg(
    unique_senders=('src', 'nunique'),
    received=('label', 'size'),
    laundering=('label', 'sum'),
)
fan_in['laundering_share'] = fan_in['laundering'] / fan_in['received']
fan_in.sort_values('unique_senders', ascending=False).head(20)

In [ ]:
bins = [0, 1, 2, 5, 10, 50, float('inf')]
fan_in['fan_in_band'] = pd.cut(fan_in['unique_senders'], bins=bins)
fan_in.groupby('fan_in_band', observed=True).agg(receivers=('received', 'size'), laundering_share=('laundering_share', 'mean'))